# step04_editor_participation — Editor Participation and Behavior

Measure contributor participation, account-type proxies, concentration, and minor-edit patterns.

Run cells from top to bottom. Shared helpers and file paths are in `analysis_common.py`. Outputs go to `analysis_results/` and `figures/`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis' / 'analysis_common.py').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis'))
from analysis_common import *
df=classify_editors(load_revisions())
per_article=[]
for article,g in df.groupby('article_title'):
    named=g.username.dropna(); counts=named.value_counts(); top=counts.head(10)
    per_article.append({'article_title':article,'revisions':len(g),'unique_usernames':named.nunique(),'unique_positive_user_ids':g.loc[g.user_id.gt(0),'user_id'].nunique(),'anonymous_revisions':int(g.editor_type.eq('Anonymous (user ID 0)').sum()),'registered_revisions':int(g.editor_type.eq('Registered (positive user ID)').sum()),'unknown_identity_revisions':int(g.editor_type.eq('Unknown / unavailable').sum()),'anonymous_pct':100*g.editor_type.eq('Anonymous (user ID 0)').mean(),'registered_pct':100*g.editor_type.eq('Registered (positive user ID)').mean(),'minor_pct':100*g.is_minor.mean(),'top1_revision_share_pct':100*top.head(1).sum()/len(g),'top5_revision_share_pct':100*top.head(5).sum()/len(g),'top10_revision_share_pct':100*top.sum()/len(g),'username_gini':gini(counts.values)})
article_stats=pd.DataFrame(per_article).sort_values('revisions',ascending=False); save_table(article_stats,'step04_article_editor_metrics.csv'); display(article_stats)


In [ ]:
editor_counts=df.groupby(['username','editor_type'],dropna=False).size().rename('revisions').reset_index().sort_values('revisions',ascending=False)
# Export top contributors without inferring identity or motive.
top=editor_counts.head(100); save_table(top,'step04_top_contributors.csv')
identity=df.editor_type.value_counts().rename_axis('editor_type').reset_index(name='revisions'); identity['share_pct']=100*identity.revisions/len(df); save_table(identity,'step04_editor_type_summary.csv'); display(identity)
fig,ax=plt.subplots(figsize=(8,5)); ax.bar(identity.editor_type,identity.share_pct,color=['#4c78a8','#f58518','#9d9d9d']); ax.set(title='Revision records by contributor metadata class',ylabel='Share of revisions (%)'); ax.tick_params(axis='x',rotation=15); finish_plot(fig,'step04_editor_types.png')
concentration=article_stats.set_index('article_title')[['top1_revision_share_pct','top5_revision_share_pct','top10_revision_share_pct']].sort_values('top10_revision_share_pct'); fig,ax=plt.subplots(figsize=(11,8)); concentration.plot.barh(ax=ax); ax.set(title='Revision share from top contributors by article',xlabel='Share of article revisions (%)'); finish_plot(fig,'step04_contributor_concentration.png')


In [ ]:
minor=(df.groupby('article_title').agg(revisions=('revision_id','size'),minor_revisions=('is_minor','sum'),minor_pct=('is_minor',lambda s:100*s.mean()),unique_editors=('username','nunique')).reset_index().sort_values('minor_pct',ascending=False)); save_table(minor,'step04_minor_edit_comparison.csv'); display(minor)
fig,ax=plt.subplots(figsize=(9,5)); ax.scatter(minor.revisions,minor.minor_pct); ax.set(title='Article activity and minor-edit share',xlabel='Revision records',ylabel='Minor edits (%)'); finish_plot(fig,'step04_minor_vs_activity.png')


## Interpretation notes

`user_id == 0` is treated as anonymous, positive IDs as registered, and missing IDs as unavailable. Username strings are not proof of a real-world identity. The Gini coefficient summarizes the distribution of revision counts across usernames within each article; it is not a measure of contributor quality.

In [ ]:
# The supplied minor-edit flag is constant in this dataset, so it cannot support comparisons.
minor_values = df.is_minor.value_counts(dropna=False)
print('is_minor values in cleaned input:', minor_values.to_dict())
if df.is_minor.nunique(dropna=False) == 1:
    print('Caution: the flag has no variation; minor-edit rates are 100% for every article and category and are not analytically discriminating.')
